<a href="https://colab.research.google.com/github/DilshanEkanayaka/AI_Course/blob/main/KVCACHE_DEMO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

model.eval()

prompt = "The red cat was"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(device)

print("Input tokens:", inputs["input_ids"].shape[1])

Device: cuda


config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Input tokens: 4


In [ ]:
def measure_generation_time(use_cache, runs=3, max_new_tokens=300):
    times = []

    # Warm-up run
    with torch.no_grad():
        _ = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            use_cache=use_cache,
            do_sample=False
        )

    if device == "cuda":
        torch.cuda.synchronize()

    # Actual benchmark
    for _ in range(runs):

        if device == "cuda":
            torch.cuda.synchronize()

        start_time = time.perf_counter()

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                use_cache=use_cache,
                do_sample=False
            )

        if device == "cuda":
            torch.cuda.synchronize()

        end_time = time.perf_counter()

        times.append(end_time - start_time)

    avg_time = sum(times) / len(times)

    return avg_time, output

In [ ]:
time_without_cache, output_without_cache = measure_generation_time(
    use_cache=False,
    runs=3,
    max_new_tokens=300
)

print(f"Average generation time WITHOUT cache: {time_without_cache:.3f} seconds")

text_without_cache = tokenizer.decode(
    output_without_cache[0],
    skip_special_tokens=True
)

print("\nGenerated text:")
print(text_without_cache)

Average generation time WITHOUT cache: 17.617 seconds

Generated text:
The red cat was a very good cat, and she was very fond of her
master.  She was always ready to do anything for him, and she was very
good to the little boy.  She would play with him, and she would bring
him his toys, and she would bring him his food.  She was very fond of
him, and he was very fond of her.  He would often sit and watch her
play, and he would often watch her bring him his food.  He would often
watch her bring him his toys.  He would often watch her bring him his
food.  He would often watch her bring him his toys.  He would often
watch her bring him his food.  He would often watch her bring him his
toys.  He would often watch her bring him his food.  He would often
watch her bring him his toys.  He would often watch her bring him his
food.  He would often watch her bring him his toys.  He would often
watch her bring him his food.  He would often watch her bring him his
toys.  He would often watch her b

In [ ]:
time_with_cache, output_with_cache = measure_generation_time(
    use_cache=True,
    runs=3,
    max_new_tokens=300
)

print(f"Average generation time WITH cache: {time_with_cache:.3f} seconds")

text_with_cache = tokenizer.decode(
    output_with_cache[0],
    skip_special_tokens=True
)

print("\nGenerated text:")
print(text_with_cache)

Average generation time WITH cache: 8.171 seconds

Generated text:
The red cat was a very good cat, and she was very fond of her
master.  She was always ready to do anything for him, and she was very
good to the little boy.  She would play with him, and she would bring
him his toys, and she would bring him his food.  She was very fond of
him, and he was very fond of her.  He would often sit and watch her
play, and he would often watch her bring him his food.  He would often
watch her bring him his toys.  He would often watch her bring him his
food.  He would often watch her bring him his toys.  He would often
watch her bring him his food.  He would often watch her bring him his
toys.  He would often watch her bring him his food.  He would often
watch her bring him his toys.  He would often watch her bring him his
food.  He would often watch her bring him his toys.  He would often
watch her bring him his food.  He would often watch her bring him his
toys.  He would often watch her bring

In [ ]:
time_saved = time_without_cache - time_with_cache

percentage_reduction = (
    time_saved / time_without_cache
) * 100

speedup = time_without_cache / time_with_cache

print("=" * 50)
print("CACHE PERFORMANCE COMPARISON")
print("=" * 50)

print(f"Without cache : {time_without_cache:.3f} sec")
print(f"With cache    : {time_with_cache:.3f} sec")
print(f"Time saved    : {time_saved:.3f} sec")
print(f"Reduction     : {percentage_reduction:.2f}%")
print(f"Speedup       : {speedup:.2f}x")

CACHE PERFORMANCE COMPARISON
Without cache : 17.617 sec
With cache    : 8.171 sec
Time saved    : 9.446 sec
Reduction     : 53.62%
Speedup       : 2.16x
